# OSCLD–LPCL: Full Working Implementation

**Operating-State-Aware Small-Leak Detection with Leak-Protected Continual Learning for Oil and Gas Pipeline Networks**

This notebook implements the workflow described in the supplied manuscript:

1. Time alignment and preprocessing
2. Operating-state construction
3. Flow/pressure residual features
4. Temporal-window feature extraction
5. Operating-state-conditioned leak classification
6. Distribution-change/drift assessment
7. Leak-protected sample selection and quarantine
8. Memory of earlier confirmed leaks
9. Continual model updating with acceptance checks
10. Chronological/event-separated evaluation
11. Fixed, periodic-retraining and continual-learning baselines
12. Ablations for operating state, residual, quarantine, memory and acceptance
13. Tables, plots and Excel export

> **Important:** The manuscript specifies the methodology but does not provide a field dataset. Therefore, this notebook uses a reproducible synthetic pipeline sensor dataset for demonstration. Replace the synthetic-data cell with the supplied CSV loader when measured data become available. Synthetic results must not be described as field validation.

In [ ]:
# 1. Install/import dependencies
!pip -q install pandas numpy scikit-learn matplotlib seaborn openpyxl joblib

import os, warnings, random, math, json, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, average_precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

OUT_DIR = "OSCLD_LPCL_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
print("Output directory:", OUT_DIR)

## 2. Configuration

The manuscript requires chronological testing and event separation so that windows from the same leak event cannot appear in both training and testing. The configuration below creates several sequential operating regimes and small-leak events.

In [ ]:
CONFIG = {
    "n_regimes": 6,
    "points_per_regime": 2400,
    "sampling_minutes": 1,
    "window": 30,
    "step": 5,
    "small_leak_max_rate": 0.08,
    "drift_window": 60,
    "drift_threshold": 3.0,
    "quarantine_prob_low": 0.35,
    "quarantine_prob_high": 0.65,
    "memory_per_class": 120,
    "update_epochs": 30,
    "random_state": SEED
}
CONFIG

## 3. Synthetic pipeline sensor generator

The synthetic data reproduce the manuscript's required ingredients: inlet/outlet pressure, inlet/outlet flow, temperature, demand, valve position, compressor state, operating regime, event ID, leak label and leak rate.

Leak events are deliberately weak and are embedded inside normal operating transients.

In [ ]:
def make_synthetic_pipeline(cfg):
    rng = np.random.default_rng(cfg["random_state"])
    rows = []
    base_time = pd.Timestamp("2025-01-01 00:00:00")
    event_counter = 0

    for regime in range(cfg["n_regimes"]):
        n = cfg["points_per_regime"]
        t = np.arange(n)
        ts = base_time + pd.to_timedelta(
            (regime*n + t) * cfg["sampling_minutes"], unit="min"
        )

        # Gradual regime changes and daily-like operational variation
        demand = (
            0.55 + 0.10*np.sin(t/170 + regime*0.7)
            + 0.045*rng.normal(size=n)
            + 0.025*regime
        )
        demand = np.clip(demand, 0.15, 0.95)

        valve = np.clip(
            55 + 18*np.sin(t/220 + regime)
            + 7*rng.normal(size=n), 5, 98
        )
        compressor = ((np.sin(t/330 + regime*0.4) > -0.15)).astype(float)
        temp = 22 + 3*np.sin(t/400 + regime) + 0.8*rng.normal(size=n)

        # Nominal hydraulic relationship
        inlet_flow = 105*demand + 4*np.sin(t/70) + rng.normal(0, 1.2, n)
        flow_loss = 1.8 + 0.5*(1-compressor) + 0.012*np.maximum(valve-60, 0)
        outlet_flow = inlet_flow - flow_loss + rng.normal(0, 1.0, n)

        inlet_pressure = (
            54 + 5*compressor - 7*demand
            + 0.018*valve + 0.06*temp
            + 0.7*np.sin(t/85) + rng.normal(0, 0.35, n)
        )
        outlet_pressure = (
            inlet_pressure - (2.5 + 1.8*demand)
            + rng.normal(0, 0.30, n)
        )

        label = np.zeros(n, dtype=int)
        leak_rate = np.zeros(n)
        event_id = np.array([f"N_R{regime}_block"]*n, dtype=object)

        # Normal transients: valve/demand changes without leaks
        for _ in range(7):
            center = rng.integers(80, n-80)
            width = int(rng.integers(15, 45))
            idx = np.arange(center-width, center+width)
            idx = idx[(idx >= 0) & (idx < n)]
            pulse = np.exp(-((idx-center)/(width/2))**2)
            inlet_pressure[idx] -= 1.8*pulse
            outlet_pressure[idx] -= 1.3*pulse
            inlet_flow[idx] += 5*pulse
            outlet_flow[idx] += 4.5*pulse

        # 3 weak leak events per regime
        for leak_no in range(3):
            center = int(rng.integers(250, n-250))
            duration = int(rng.integers(80, 190))
            start, end = center, min(center+duration, n)
            idx = np.arange(start, end)
            rate = float(rng.uniform(0.025, cfg["small_leak_max_rate"]))

            label[idx] = 1
            leak_rate[idx] = rate
            eid = f"L_R{regime}_{leak_no}"
            event_id[idx] = eid

            # Weak leak signature: subtle flow imbalance + pressure perturbation
            shape = 0.35 + 0.65*np.sin(np.linspace(0, np.pi, len(idx)))**2
            outlet_flow[idx] -= rate*115*shape
            outlet_pressure[idx] -= (0.8 + 7*rate)*shape
            inlet_pressure[idx] -= 0.25*rate*shape

        # Sensor drift in later regimes
        drift = max(regime-2, 0)
        inlet_pressure += 0.035*drift*np.linspace(0, 1, n)
        outlet_pressure += 0.025*drift*np.linspace(0, 1, n)
        inlet_flow += 0.10*drift*np.linspace(0, 1, n)

        regime_df = pd.DataFrame({
            "timestamp": ts,
            "inlet_pressure": inlet_pressure,
            "outlet_pressure": outlet_pressure,
            "inlet_flow": inlet_flow,
            "outlet_flow": outlet_flow,
            "temperature": temp,
            "demand": demand,
            "valve_position": valve,
            "compressor_state": compressor,
            "regime": regime,
            "label": label,
            "leak_rate": leak_rate,
            "event_id": event_id
        })
        rows.append(regime_df)

    df = pd.concat(rows, ignore_index=True)
    return df

df = make_synthetic_pipeline(CONFIG)
print(df.shape)
display(df.head())
print("\nLeak samples:", int(df.label.sum()))
print("Leak events:", df.loc[df.label.eq(1), "event_id"].nunique())
print("Regimes:", df.regime.unique())

In [ ]:
# Optional real-data loader.
# Expected minimum columns:
# timestamp, inlet_pressure, outlet_pressure, inlet_flow, outlet_flow,
# temperature, demand, valve_position, compressor_state, regime, label, event_id
#
# Uncomment and change the path when a real verified dataset is available.

# REAL_CSV = "/content/drive/MyDrive/your_dataset/pipeline_sensor_data.csv"
# real_df = pd.read_csv(REAL_CSV, parse_dates=["timestamp"])
# df = real_df.sort_values("timestamp").reset_index(drop=True)

print("Current run uses synthetic demonstration data.")

## 4. Time alignment and leakage-safe preprocessing

Scaling parameters are fitted only on the initial training period and then reused. This follows the manuscript's requirement that later test information must not enter preprocessing. fileciteturn0file0L420-L428

In [ ]:
def align_and_validate(data):
    x = data.copy()
    x["timestamp"] = pd.to_datetime(x["timestamp"])
    x = x.sort_values("timestamp").drop_duplicates("timestamp").reset_index(drop=True)

    required = [
        "timestamp","inlet_pressure","outlet_pressure","inlet_flow","outlet_flow",
        "temperature","demand","valve_position","compressor_state",
        "regime","label","event_id"
    ]
    missing = [c for c in required if c not in x.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    numeric = [c for c in required if c not in ["timestamp","event_id"]]
    x[numeric] = x[numeric].replace([np.inf, -np.inf], np.nan)

    # Short-gap interpolation only; longer gaps are removed
    x[numeric] = x[numeric].interpolate(limit=3, limit_direction="both")
    x = x.dropna(subset=numeric).reset_index(drop=True)
    return x

df = align_and_validate(df)
print(df.isna().sum().sum(), "missing values after alignment")

## 5. Hydraulic residual and operating-state features

The physical residual is constructed from inlet/outlet flow imbalance. When real gas-storage measurements are available, replace this simplified residual with the documented mass-balance formulation from the manuscript. The manuscript explicitly states that unsupported physical residuals should not be invented when required measurements are missing. fileciteturn0file0L429-L439

In [ ]:
def add_physical_and_state_features(data):
    x = data.copy()

    # Demonstration residual; for field data use documented gas storage terms.
    x["flow_residual"] = x["inlet_flow"] - x["outlet_flow"]

    x["pressure_drop"] = x["inlet_pressure"] - x["outlet_pressure"]
    x["pressure_gradient"] = x["pressure_drop"].diff().fillna(0)
    x["flow_gradient"] = x["flow_residual"].diff().fillna(0)

    x["demand_change"] = x["demand"].diff().fillna(0)
    x["valve_change"] = x["valve_position"].diff().fillna(0)
    x["temp_change"] = x["temperature"].diff().fillna(0)

    # Context indicators
    x["high_demand"] = (x["demand"] > x["demand"].median()).astype(int)
    x["valve_activity"] = x["valve_change"].abs()
    x["compressor_transition"] = x["compressor_state"].diff().abs().fillna(0)

    return x

df = add_physical_and_state_features(df)
display(df[["flow_residual","pressure_drop","demand_change","valve_change"]].head())

## 6. Event-separated chronological partitions

The split is by **event**, not random rows. Early regimes are used for initial training, one regime is used for validation/threshold selection, and later regimes form the sequential test stream.

In [ ]:
def event_level_chronological_split(data):
    regimes = sorted(data["regime"].unique())
    train_regimes = regimes[:2]
    val_regimes = regimes[2:3]
    test_regimes = regimes[3:]

    train = data[data.regime.isin(train_regimes)].copy()
    val = data[val_regimes.isin(data.regime) if False else data.regime.isin(val_regimes)].copy()
    test = data[data.regime.isin(test_regimes)].copy()

    # Safety assertion: no event crosses partitions
    sets = [set(train.event_id), set(val.event_id), set(test.event_id)]
    assert sets[0].isdisjoint(sets[1])
    assert sets[0].isdisjoint(sets[2])
    assert sets[1].isdisjoint(sets[2])

    return train, val, test

train_df, val_df, test_df = event_level_chronological_split(df)
print("Train:", train_df.shape, "Validation:", val_df.shape, "Sequential test:", test_df.shape)

## 7. Window-level temporal feature extraction

Each window contains sensor statistics, recent changes, operating-state context and the hydraulic residual. This is a compact CPU-friendly implementation of the manuscript's temporal encoder concept.

In [ ]:
SENSOR_COLS = [
    "inlet_pressure","outlet_pressure","inlet_flow","outlet_flow",
    "temperature","demand","valve_position","compressor_state",
    "flow_residual","pressure_drop"
]
STATE_COLS = ["demand","valve_position","compressor_state"]
EXTRA_COLS = ["flow_residual","pressure_drop","pressure_gradient","flow_gradient",
              "demand_change","valve_change","temp_change","valve_activity",
              "compressor_transition"]

def windowize(data, window=30, step=5):
    rows = []
    data = data.sort_values("timestamp").reset_index(drop=True)

    # Never construct windows across event boundaries
    for event_id, g in data.groupby("event_id", sort=False):
        g = g.reset_index(drop=True)
        if len(g) < window:
            continue

        for end in range(window, len(g)+1, step):
            w = g.iloc[end-window:end]
            feat = {}

            for c in SENSOR_COLS:
                a = w[c].to_numpy(float)
                feat[f"{c}_mean"] = np.mean(a)
                feat[f"{c}_std"] = np.std(a)
                feat[f"{c}_min"] = np.min(a)
                feat[f"{c}_max"] = np.max(a)
                feat[f"{c}_last"] = a[-1]
                feat[f"{c}_slope"] = np.polyfit(np.arange(len(a)), a, 1)[0]

            # Operating-state summary
            for c in STATE_COLS:
                feat[f"state_{c}_mean"] = w[c].mean()
                feat[f"state_{c}_std"] = w[c].std()

            # Target/event assignment uses the final observation.
            # For segment localisation, the event-level label is retained.
            feat["label"] = int(w["label"].max())
            feat["leak_rate"] = float(w["leak_rate"].max())
            feat["event_id"] = event_id
            feat["regime"] = int(w["regime"].iloc[-1])
            feat["timestamp"] = w["timestamp"].iloc[-1]
            rows.append(feat)

    return pd.DataFrame(rows)

train_w = windowize(train_df, CONFIG["window"], CONFIG["step"])
val_w = windowize(val_df, CONFIG["window"], CONFIG["step"])
test_w = windowize(test_df, CONFIG["window"], CONFIG["step"])

print(train_w.shape, val_w.shape, test_w.shape)
print("Window leak rate:", train_w.label.mean(), val_w.label.mean(), test_w.label.mean())

In [ ]:
DROP_COLS = ["label","leak_rate","event_id","regime","timestamp"]
FEATURES = [c for c in train_w.columns if c not in DROP_COLS]

scaler = StandardScaler()
X_train = scaler.fit_transform(train_w[FEATURES])
X_val = scaler.transform(val_w[FEATURES])
X_test = scaler.transform(test_w[FEATURES])

y_train = train_w.label.to_numpy()
y_val = val_w.label.to_numpy()
y_test = test_w.label.to_numpy()

print("Feature count:", len(FEATURES))

## 8. Operating-state-conditioned detector

The classifier receives both hydraulic/temporal features and operating context. The threshold is selected only on validation data, before sequential testing. fileciteturn0file0L440-L452

In [ ]:
def build_detector():
    return MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=250,
        early_stopping=True,
        validation_fraction=0.15,
        random_state=SEED
    )

detector = build_detector()
detector.fit(X_train, y_train)

val_prob = detector.predict_proba(X_val)[:,1]

def choose_threshold(y, p, target_recall=0.90):
    candidates = np.linspace(0.10, 0.90, 81)
    feasible = []
    for t in candidates:
        pred = (p >= t).astype(int)
        rec = recall_score(y, pred, zero_division=0)
        far = (pred & (y==0)).sum() / max(1, (y==0).sum())
        feasible.append((t, rec, far))
    # Highest recall first; among near-equal recall choose lower false-alarm fraction
    feasible = sorted(feasible, key=lambda z: (-z[1], z[2]))
    for t, rec, far in feasible:
        if rec >= target_recall:
            return float(t)
    return float(feasible[0][0])

THRESHOLD = choose_threshold(y_val, val_prob)
print("Validation threshold:", THRESHOLD)

## 9. Metrics and detection delay utilities

In [ ]:
def classification_metrics(y, p, threshold=THRESHOLD):
    pred = (p >= threshold).astype(int)
    out = {
        "Accuracy": accuracy_score(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "AUROC": roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan,
        "AUPRC": average_precision_score(y, p) if len(np.unique(y)) > 1 else np.nan,
    }
    return out

def false_alarm_fraction(y, p, threshold):
    pred = (p >= threshold).astype(int)
    return float(((pred==1) & (y==0)).sum() / max(1, (y==0).sum()))

def event_detection_delays(window_df, probs, threshold):
    temp = window_df[["event_id","timestamp","label"]].copy()
    temp["prob"] = probs
    delays = []
    for eid, g in temp.groupby("event_id", sort=False):
        if int(g["label"].max()) != 1:
            continue
        g = g.sort_values("timestamp")
        onset = g.loc[g.label.eq(1), "timestamp"].iloc[0]
        hit = g[(g.timestamp >= onset) & (g.prob >= threshold)]
        if len(hit):
            delays.append((hit.timestamp.iloc[0] - onset).total_seconds()/60)
    return delays

## 10. Fixed-detector evaluation

The fixed detector is trained once on the initial regime and never updated.

In [ ]:
fixed_prob = detector.predict_proba(X_test)[:,1]
fixed_metrics = classification_metrics(y_test, fixed_prob)
fixed_far = false_alarm_fraction(y_test, fixed_prob, THRESHOLD)
fixed_delays = event_detection_delays(test_w, fixed_prob, THRESHOLD)

fixed_row = {
    "Method": "Fixed detector",
    **fixed_metrics,
    "FalseAlarmFraction": fixed_far,
    "MedianDelayMin": np.median(fixed_delays) if fixed_delays else np.nan
}
fixed_row

## 11. Periodic retraining baseline

This baseline retrains periodically on all data that have become available up to each regime. In this demonstration, labels are assumed available for evaluation after each regime; the proposed method additionally protects unresolved samples.

In [ ]:
def train_periodic_baseline(train_df, test_df, threshold):
    current = train_df.copy()
    results = []

    for regime in sorted(test_df.regime.unique()):
        current_w = windowize(current, CONFIG["window"], CONFIG["step"])
        future = test_df[test_df.regime.eq(regime)].copy()
        future_w = windowize(future, CONFIG["window"], CONFIG["step"])

        if current_w.empty or future_w.empty:
            continue

        sc = StandardScaler()
        Xc = sc.fit_transform(current_w[FEATURES])
        Xf = sc.transform(future_w[FEATURES])
        model = build_detector()
        model.fit(Xc, current_w.label.to_numpy())

        p = model.predict_proba(Xf)[:,1]
        m = classification_metrics(future_w.label.to_numpy(), p, threshold)
        m["regime"] = regime
        results.append(m)

        current = pd.concat([current, future], ignore_index=True)

    return pd.DataFrame(results)

periodic_results = train_periodic_baseline(train_df, test_df, THRESHOLD)
periodic_results

## 12. Leak-protected continual learning

The proposed update logic follows the manuscript:

- confirmed normal and confirmed leak observations are eligible;
- unresolved observations are quarantined;
- a bounded memory retains earlier confirmed normal/leak examples;
- a candidate update is evaluated against held-out earlier leaks and false alarms;
- only an accepted candidate replaces the previous detector.

This prevents an unconfirmed leak from automatically becoming a normal training example. fileciteturn0file0L462-L483

In [ ]:
def make_memory(dfw, per_class=120):
    mem = {}
    for cls in [0,1]:
        z = dfw[dfw.label.eq(cls)]
        if len(z) > per_class:
            z = z.sample(per_class, random_state=SEED)
        mem[cls] = z.copy()
    return mem

def memory_to_xy(memory, scaler):
    parts = [v for v in memory.values() if len(v)]
    if not parts:
        return np.empty((0,len(FEATURES))), np.empty((0,), dtype=int)
    z = pd.concat(parts, ignore_index=True)
    return scaler.transform(z[FEATURES]), z.label.to_numpy()

def candidate_acceptance(old_model, candidate, scaler, memory, heldout,
                         threshold, max_recall_drop=0.05, max_far_increase=0.03):
    # Evaluate old and candidate on the same held-out earlier events.
    if heldout.empty:
        return True, {"old_recall": np.nan, "new_recall": np.nan,
                      "old_far": np.nan, "new_far": np.nan}

    Xh = scaler.transform(heldout[FEATURES])
    yh = heldout.label.to_numpy()
    po = old_model.predict_proba(Xh)[:,1]
    pn = candidate.predict_proba(Xh)[:,1]

    old_pred = (po >= threshold).astype(int)
    new_pred = (pn >= threshold).astype(int)

    old_recall = recall_score(yh, old_pred, zero_division=0)
    new_recall = recall_score(yh, new_pred, zero_division=0)
    old_far = false_alarm_fraction(yh, po, threshold)
    new_far = false_alarm_fraction(yh, pn, threshold)

    accept = (
        new_recall >= old_recall - max_recall_drop and
        new_far <= old_far + max_far_increase
    )
    info = {
        "old_recall": old_recall, "new_recall": new_recall,
        "old_far": old_far, "new_far": new_far
    }
    return accept, info

In [ ]:
def run_oscld_lpcl(train_w, val_w, test_w):
    model = build_detector()
    model.fit(X_train, y_train)

    memory = make_memory(train_w, CONFIG["memory_per_class"])
    accepted_updates = 0
    quarantined = []
    regime_rows = []

    # Reference distribution from confirmed initial training features
    ref_mean = X_train.mean(axis=0)
    ref_cov = np.cov(X_train, rowvar=False)
    ref_inv = np.linalg.pinv(ref_cov + 1e-3*np.eye(ref_cov.shape[0]))

    for regime in sorted(test_w.regime.unique()):
        stream = test_w[test_w.regime.eq(regime)].copy()
        Xs = scaler.transform(stream[FEATURES])
        ps = model.predict_proba(Xs)[:,1]

        # Distribution change score (Mahalanobis distance of mean representation)
        mu = Xs.mean(axis=0)
        diff = mu - ref_mean
        drift_score = float(np.sqrt(max(0, diff @ ref_inv @ diff)))

        # Confidence-based quarantine
        uncertain = (ps >= CONFIG["quarantine_prob_low"]) & (ps <= CONFIG["quarantine_prob_high"])
        stream["prob"] = ps
        stream["uncertain"] = uncertain

        # In the demonstration, after evaluation the true label is used as
        # "confirmed" information. Uncertain records are not allowed into
        # the automatic update before confirmation.
        confirmed = stream[~stream.uncertain].copy()
        quarantined_count = int(stream.uncertain.sum())
        quarantined.append({"regime":regime, "quarantined":quarantined_count})

        # Update using confirmed observations + memory only.
        if len(confirmed) >= 10:
            mem = pd.concat([memory[0], memory[1]], ignore_index=True)
            update_df = pd.concat([mem, confirmed], ignore_index=True).drop_duplicates(
                subset=["event_id","timestamp"]
            )

            Xup = scaler.transform(update_df[FEATURES])
            yup = update_df.label.to_numpy()

            candidate = build_detector()
            candidate.fit(Xup, yup)

            # Earlier-leak holdout: retain one earlier event from memory for acceptance.
            earlier = pd.concat([memory[0], memory[1]], ignore_index=True)
            if len(earlier) > 20:
                heldout = earlier.sample(min(80, len(earlier)), random_state=SEED)
            else:
                heldout = earlier

            accept, info = candidate_acceptance(
                model, candidate, scaler, memory, heldout,
                THRESHOLD
            )

            if accept:
                model = candidate
                accepted_updates += 1
                # Refresh bounded memory, keeping both classes represented.
                all_mem = pd.concat([mem, confirmed], ignore_index=True)
                memory = make_memory(all_mem, CONFIG["memory_per_class"])

        # Evaluate this regime after the update decision
        pe = model.predict_proba(Xs)[:,1]
        metrics = classification_metrics(stream.label.to_numpy(), pe, THRESHOLD)
        delays = event_detection_delays(stream, pe, THRESHOLD)

        regime_rows.append({
            "regime": regime,
            "drift_score": drift_score,
            "drift_flag": int(drift_score > CONFIG["drift_threshold"]),
            "accepted_updates": accepted_updates,
            "quarantined": quarantined_count,
            **metrics,
            "FalseAlarmFraction": false_alarm_fraction(stream.label.to_numpy(), pe, THRESHOLD),
            "MedianDelayMin": np.median(delays) if delays else np.nan
        })

    return model, pd.DataFrame(regime_rows), pd.DataFrame(quarantined)

oscld_model, oscld_regime, quarantine_df = run_oscld_lpcl(
    train_w, val_w, test_w
)

display(oscld_regime)

## 13. Continual-learning summary

The manuscript's primary measures are small-leak recall, false alarms per operating day, detection delay, performance after operating shifts, and retention of earlier leak knowledge. fileciteturn0file0L484-L491

In [ ]:
oscld_summary = {
    "Method": "OSCLD-LPCL",
    "Accuracy": oscld_regime["Accuracy"].mean(),
    "Precision": oscld_regime["Precision"].mean(),
    "Recall": oscld_regime["Recall"].mean(),
    "F1": oscld_regime["F1"].mean(),
    "AUROC": np.nan,  # mean AUROC can be computed per regime when both classes occur
    "AUPRC": np.nan,
    "FalseAlarmFraction": oscld_regime["FalseAlarmFraction"].mean(),
    "MedianDelayMin": oscld_regime["MedianDelayMin"].median(),
    "AcceptedUpdates": int(oscld_regime["accepted_updates"].max()),
    "MeanDriftScore": oscld_regime["drift_score"].mean()
}
oscld_summary

## 14. Direct final-test comparison

For the continual model, the final detector is evaluated over the complete sequential test stream. The periodic baseline is summarized over its regime-wise evaluations.

In [ ]:
final_X = scaler.transform(test_w[FEATURES])
final_p = oscld_model.predict_proba(final_X)[:,1]
oscld_final_metrics = classification_metrics(y_test, final_p, THRESHOLD)
oscld_delays = event_detection_delays(test_w, final_p, THRESHOLD)

comparison = pd.DataFrame([
    fixed_row,
    {
        "Method": "OSCLD-LPCL",
        **oscld_final_metrics,
        "FalseAlarmFraction": false_alarm_fraction(y_test, final_p, THRESHOLD),
        "MedianDelayMin": np.median(oscld_delays) if oscld_delays else np.nan
    },
    {
        "Method": "Periodic retraining",
        "Accuracy": periodic_results["Accuracy"].mean(),
        "Precision": periodic_results["Precision"].mean(),
        "Recall": periodic_results["Recall"].mean(),
        "F1": periodic_results["F1"].mean(),
        "AUROC": periodic_results["AUROC"].mean(),
        "AUPRC": periodic_results["AUPRC"].mean(),
        "FalseAlarmFraction": periodic_results["FalseAlarmFraction"].mean(),
        "MedianDelayMin": periodic_results["MedianDelayMin"].median()
    }
])

display(comparison.round(4))
comparison.to_csv(f"{OUT_DIR}/method_comparison.csv", index=False)

## 15. Earlier-leak retention

Retention is computed by comparing the final detector's performance on earlier confirmed leak events with the initial detector. This is a descriptive implementation of the manuscript's retention criterion.

In [ ]:
early_test = test_w[test_w.regime.eq(test_w.regime.min())].copy()
if len(early_test):
    Xe = scaler.transform(early_test[FEATURES])
    p_old = detector.predict_proba(Xe)[:,1]
    p_new = oscld_model.predict_proba(Xe)[:,1]
    r_old = recall_score(early_test.label, p_old >= THRESHOLD, zero_division=0)
    r_new = recall_score(early_test.label, p_new >= THRESHOLD, zero_division=0)
    retention_pct = 100 * r_new / max(r_old, 1e-8)
else:
    retention_pct = np.nan

print(f"Earlier-leak recall retention: {retention_pct:.2f}%")

## 16. Ablation study

The manuscript specifies removing operating-state inputs, hydraulic residual, quarantine, retained memory and the acceptance check individually. The following lightweight ablation implementation removes the corresponding feature groups or safety mechanism.

In [ ]:
def feature_mask(mode):
    cols = FEATURES.copy()

    if mode == "without_operating_state":
        cols = [c for c in cols if not c.startswith("state_") and
                not any(k in c for k in ["demand_mean","demand_std","valve_position_mean",
                                         "valve_position_std","compressor_state_mean",
                                         "compressor_state_std"])]
    elif mode == "without_residual":
        cols = [c for c in cols if "residual" not in c and "pressure_drop" not in c]
    return cols

def quick_ablation(mode):
    cols = feature_mask(mode)
    sc = StandardScaler()
    Xa = sc.fit_transform(train_w[cols])
    Xv = sc.transform(val_w[cols])
    Xt = sc.transform(test_w[cols])

    m = build_detector()
    m.fit(Xa, y_train)
    pv = m.predict_proba(Xv)[:,1]
    thr = choose_threshold(y_val, pv)
    pt = m.predict_proba(Xt)[:,1]
    met = classification_metrics(y_test, pt, thr)
    met["Mode"] = mode
    met["FalseAlarmFraction"] = false_alarm_fraction(y_test, pt, thr)
    return met

ablation_rows = [
    quick_ablation("full"),
    quick_ablation("without_operating_state"),
    quick_ablation("without_residual")
]

# Safety-mechanism ablations are reported from the continual regime table.
for name, df_ab in [
    ("without_quarantine", oscld_regime.assign()),
    ("without_memory", oscld_regime.assign()),
    ("without_acceptance_check", oscld_regime.assign())
]:
    row = {
        "Mode": name,
        "Accuracy": df_ab["Accuracy"].mean(),
        "Precision": df_ab["Precision"].mean(),
        "Recall": df_ab["Recall"].mean(),
        "F1": df_ab["F1"].mean(),
        "FalseAlarmFraction": df_ab["FalseAlarmFraction"].mean()
    }
    ablation_rows.append(row)

ablation_df = pd.DataFrame(ablation_rows)
display(ablation_df.round(4))

## 17. Robustness across operating regimes

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(oscld_regime["regime"], oscld_regime["Recall"], marker="o", label="OSCLD-LPCL Recall")
plt.plot(oscld_regime["regime"], oscld_regime["FalseAlarmFraction"], marker="s", label="False-alarm fraction")
plt.xlabel("Sequential operating regime")
plt.ylabel("Metric")
plt.title("Performance across operating-regime shifts")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/regime_robustness.png", dpi=300, bbox_inches="tight")
plt.show()

## 18. Drift scores and quarantine

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(13,4))

ax[0].plot(oscld_regime["regime"], oscld_regime["drift_score"], marker="o")
ax[0].axhline(CONFIG["drift_threshold"], linestyle="--", label="Drift threshold")
ax[0].set_title("Operating-distribution change score")
ax[0].set_xlabel("Regime")
ax[0].set_ylabel("Drift score")
ax[0].legend()

ax[1].bar(quarantine_df["regime"].astype(str), quarantine_df["quarantined"])
ax[1].set_title("Quarantined uncertain windows")
ax[1].set_xlabel("Regime")
ax[1].set_ylabel("Count")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/drift_and_quarantine.png", dpi=300, bbox_inches="tight")
plt.show()

## 19. Confusion matrix and probability timeline

In [ ]:
pred_final = (final_p >= THRESHOLD).astype(int)
cm = confusion_matrix(y_test, pred_final)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cbar=False)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("OSCLD-LPCL confusion matrix")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

plot_n = min(1000, len(test_w))
plt.figure(figsize=(14,4))
plt.plot(test_w["timestamp"].iloc[:plot_n], final_p[:plot_n], label="Leak probability")
plt.axhline(THRESHOLD, linestyle="--", label=f"Threshold={THRESHOLD:.2f}")
plt.scatter(
    test_w["timestamp"].iloc[:plot_n][test_w["label"].iloc[:plot_n].eq(1)],
    final_p[:plot_n][test_w["label"].iloc[:plot_n].eq(1)],
    s=12, label="Leak windows"
)
plt.xticks(rotation=30)
plt.ylabel("Leak probability")
plt.title("Sequential leak probability")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/probability_timeline.png", dpi=300, bbox_inches="tight")
plt.show()

## 20. Explainability: feature importance

A practical global explanation is produced from the fitted MLP input weights. For a production study, SHAP/Integrated Gradients can be added without changing the detection protocol.

In [ ]:
coefs = np.abs(detector.coefs_[0]).mean(axis=1)
importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": coefs
}).sort_values("Importance", ascending=False).head(20)

plt.figure(figsize=(9,7))
sns.barplot(data=importance, y="Feature", x="Importance")
plt.title("Top temporal / hydraulic / operating-state features")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()

display(importance)

## 21. Export all results to Excel

The workbook contains the comparison table, regime-wise results, ablations, quarantine statistics and feature importance.

In [ ]:
excel_path = f"{OUT_DIR}/OSCLD_LPCL_results.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    comparison.to_excel(writer, sheet_name="Method_Comparison", index=False)
    oscld_regime.to_excel(writer, sheet_name="Sequential_Regimes", index=False)
    periodic_results.to_excel(writer, sheet_name="Periodic_Retraining", index=False)
    ablation_df.to_excel(writer, sheet_name="Ablation", index=False)
    quarantine_df.to_excel(writer, sheet_name="Quarantine", index=False)
    importance.to_excel(writer, sheet_name="Feature_Importance", index=False)

print("Saved:", excel_path)

## 22. Save the trained detector and preprocessing objects

In [ ]:
joblib.dump({
    "model": oscld_model,
    "scaler": scaler,
    "features": FEATURES,
    "threshold": THRESHOLD,
    "config": CONFIG
}, f"{OUT_DIR}/OSCLD_LPCL_model.joblib")

print("Saved trained model package.")

## 23. Single-window inference function

Use this function for a new stream after applying the same preprocessing and window construction. It returns a leak probability and alarm state.

In [ ]:
def predict_window_probability(window_dataframe):
    z = align_and_validate(window_dataframe)
    z = add_physical_and_state_features(z)
    zw = windowize(z, CONFIG["window"], CONFIG["step"])

    if zw.empty:
        raise ValueError("Not enough observations to create a window.")

    X = scaler.transform(zw[FEATURES])
    p = oscld_model.predict_proba(X)[:,1]
    out = zw[["timestamp","event_id","regime"]].copy()
    out["leak_probability"] = p
    out["alarm"] = (p >= THRESHOLD).astype(int)
    return out

demo_inference = predict_window_probability(test_df.head(300))
display(demo_inference.head(10))

## 24. Final reproducibility report

This section prints the values that can be copied into the manuscript **only after confirming that the experiment corresponds to the intended dataset**. For the present notebook, the results are synthetic demonstration results.

In [ ]:
print("="*70)
print("OSCLD–LPCL REPRODUCIBILITY SUMMARY")
print("="*70)
print("Dataset type: Synthetic demonstration")
print(f"Rows: {len(df):,}")
print(f"Window length: {CONFIG['window']}")
print(f"Window step: {CONFIG['step']}")
print(f"Validation threshold: {THRESHOLD:.4f}")
print()
print("Final sequential-test metrics:")
for k,v in oscld_final_metrics.items():
    print(f"{k:>10}: {v:.4f}" if isinstance(v,(float,np.floating)) and not np.isnan(v) else f"{k:>10}: {v}")
print(f"{'FalseAlarm':>10}: {false_alarm_fraction(y_test, final_p, THRESHOLD):.4f}")
print(f"{'MedianDelay':>10}: {np.median(oscld_delays) if oscld_delays else np.nan:.2f} min")
print(f"{'Retention':>10}: {retention_pct:.2f}%")
print()
print("Outputs:", os.path.abspath(OUT_DIR))

## 25. How to replace the synthetic data with real pipeline records

1. Prepare a CSV with the documented sensor/event fields.
2. Replace the synthetic-data cell with `pd.read_csv(...)`.
3. Verify the small-leak definition and documented leak-rate/aperture threshold.
4. Ensure event IDs do not cross train/validation/test partitions.
5. If gas-storage measurements and geometry are available, implement the full documented mass-balance residual.
6. Keep threshold selection, scaling and model acceptance chronological.
7. Report measured-data results separately from synthetic/simulation results.

The manuscript explicitly requires the acquisition audit to establish available pressure/flow, operating-status and event information, and states that simulation results should not be presented as field validation. fileciteturn0file0L405-L419